# 02 — Results Overview

Visualize performance across all model/dataset combinations: heatmaps, bar charts, rank analyses, win/loss/tie matrices, and summary tables.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'savefig.dpi': 300,
})

SAVE_DPI = 300
os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

# Load primary metrics from config
with open('../configs/experiment.yaml', 'r') as f:
    exp_cfg = yaml.safe_load(f)

PRIMARY_METRICS = {
    'binary': exp_cfg['primary_metric_binary'],       # roc_auc
    'multiclass': exp_cfg['primary_metric_multiclass'], # log_loss
    'regression': exp_cfg['primary_metric_regression'], # rmse
}
# For multiclass log_loss and regression rmse: lower is better
HIGHER_IS_BETTER = {
    'binary': True,
    'multiclass': False,  # log_loss: lower is better
    'regression': False,  # rmse: lower is better
}

print("Primary metrics:", PRIMARY_METRICS)

In [ ]:
# Load aggregated results — graceful handling if files not yet generated
try:
    test_df = pd.read_csv('../results/aggregated/test_results.csv')
    fold_df = pd.read_csv('../results/aggregated/fold_results.csv')
    print(f"Loaded test results: {test_df.shape}, fold results: {fold_df.shape}")
    print("Columns in test_df:", test_df.columns.tolist())
    test_df.head()
except FileNotFoundError as e:
    print(f"Results not yet available: {e}")
    print("Run: python scripts/run_all.py && python scripts/evaluate.py")
    test_df = pd.DataFrame()
    fold_df = pd.DataFrame()

## Performance Heatmaps

Heatmaps using the primary metric for each task type: `roc_auc` for binary, `log_loss` for multiclass, `rmse` for regression (both lower-is-better metrics rendered with reversed colormaps).

In [ ]:
if not test_df.empty:
    # Classification heatmap — roc_auc for binary, log_loss for multiclass
    clf_df = test_df[test_df['task_type'].isin(['binary', 'multiclass'])].copy()

    def get_primary_metric_value(row):
        metric = PRIMARY_METRICS[row['task_type']]
        return row.get(metric, np.nan)

    clf_df['primary_metric'] = clf_df.apply(get_primary_metric_value, axis=1)
    pivot_clf = clf_df.pivot(index='dataset', columns='model', values='primary_metric')

    plt.figure(figsize=(12, 5))
    # Use diverging colormap: for mixed binary/multiclass, annotate metric per dataset
    sns.heatmap(pivot_clf, annot=True, fmt='.3f', cmap='YlGn', linewidths=0.5)
    plt.title('Classification Performance\n(roc_auc for binary [higher=better], log_loss for multiclass [lower=better])')
    plt.tight_layout()
    plt.savefig('../results/figures/heatmap_classification.png', dpi=SAVE_DPI, bbox_inches='tight')
    plt.savefig('../results/figures/heatmap_classification.pdf', bbox_inches='tight')
    plt.show()
else:
    print("Skipping: results not available.")

In [ ]:
if not test_df.empty:
    # Regression heatmap — normalized RMSE (RMSE / std(y_test)) for cross-dataset comparability
    reg_df = test_df[test_df['task_type'] == 'regression'].copy()

    if 'rmse' in reg_df.columns and 'y_std' in reg_df.columns:
        # Normalize by target std if available
        reg_df['nrmse'] = reg_df['rmse'] / reg_df['y_std']
        metric_col = 'nrmse'
        metric_label = 'Normalized RMSE (RMSE / std(y)) — lower is better'
    else:
        # Fall back to rank-normalized RMSE for cross-dataset comparability
        pivot_raw = reg_df.pivot(index='dataset', columns='model', values='rmse')
        # Rank within each dataset (rank 1 = best = lowest RMSE)
        pivot_ranks = pivot_raw.rank(axis=1, ascending=True)
        plt.figure(figsize=(12, 4))
        sns.heatmap(pivot_ranks, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
        plt.title('Regression Performance — Per-Dataset Rank (1=best, lower RMSE)')
        plt.tight_layout()
        plt.savefig('../results/figures/heatmap_regression_ranks.png', dpi=SAVE_DPI, bbox_inches='tight')
        plt.savefig('../results/figures/heatmap_regression_ranks.pdf', bbox_inches='tight')
        plt.show()
        metric_col = 'rmse'
        metric_label = 'RMSE — lower is better'

    pivot_reg = reg_df.pivot(index='dataset', columns='model', values=metric_col)

    plt.figure(figsize=(12, 4))
    sns.heatmap(pivot_reg, annot=True, fmt='.3f', cmap='YlOrRd_r', linewidths=0.5)
    plt.title(f'Regression Performance ({metric_label})')
    plt.tight_layout()
    plt.savefig('../results/figures/heatmap_regression.png', dpi=SAVE_DPI, bbox_inches='tight')
    plt.savefig('../results/figures/heatmap_regression.pdf', bbox_inches='tight')
    plt.show()
else:
    print("Skipping: results not available.")

## Rank-Based Overview

Computing per-dataset ranks and aggregating to average ranks — a scale-invariant comparison across datasets with different metrics.

In [ ]:
if not test_df.empty:
    tasks_config = [
        ('binary', PRIMARY_METRICS['binary'], HIGHER_IS_BETTER['binary']),
        ('multiclass', PRIMARY_METRICS['multiclass'], HIGHER_IS_BETTER['multiclass']),
        ('regression', PRIMARY_METRICS['regression'], HIGHER_IS_BETTER['regression']),
    ]

    all_rank_frames = []

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, (task_type, metric, higher) in zip(axes, tasks_config):
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            ax.set_visible(False)
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
        if pivot.empty:
            ax.set_visible(False)
            continue

        # Rank within each dataset (ascending=not higher: rank 1 = best)
        ranks = pivot.rank(axis=1, ascending=not higher)
        avg_ranks = ranks.mean(axis=0).sort_values()
        all_rank_frames.append(avg_ranks.rename(task_type))

        colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(avg_ranks)))
        bars = ax.barh(avg_ranks.index, avg_ranks.values, color=colors)
        ax.set_xlabel('Average Rank (lower = better)')
        ax.set_title(f'{task_type.title()}\n({metric})')
        ax.invert_yaxis()
        for bar, val in zip(bars, avg_ranks.values):
            ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
                    f'{val:.2f}', va='center', fontsize=9)

    plt.suptitle('Average Rank per Task Type (Rank 1 = Best)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../results/figures/average_ranks.png', dpi=SAVE_DPI, bbox_inches='tight')
    plt.savefig('../results/figures/average_ranks.pdf', bbox_inches='tight')
    plt.show()

    # Combined rank table
    if all_rank_frames:
        rank_table = pd.concat(all_rank_frames, axis=1)
        rank_table['Overall Avg'] = rank_table.mean(axis=1)
        rank_table = rank_table.sort_values('Overall Avg')
        print("\nAverage ranks by task type:")
        display(rank_table.round(2).style.set_caption('Average Ranks (lower = better)').background_gradient(cmap='RdYlGn_r', axis=0))
else:
    print("Skipping: results not available.")

## Win/Loss/Tie Matrix

For each pair of models, count the number of datasets where model A outperforms model B (win), underperforms (loss), or performs similarly within tolerance (tie). A symmetric matrix is shown as a heatmap.

In [ ]:
if not test_df.empty:
    TOLERANCE = 1e-4  # tolerance for tie determination

    tasks_config = [
        ('binary', PRIMARY_METRICS['binary'], HIGHER_IS_BETTER['binary']),
        ('multiclass', PRIMARY_METRICS['multiclass'], HIGHER_IS_BETTER['multiclass']),
        ('regression', PRIMARY_METRICS['regression'], HIGHER_IS_BETTER['regression']),
    ]

    for task_type, metric, higher in tasks_config:
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
        if pivot.shape[1] < 2:
            continue

        models = pivot.columns.tolist()
        n = len(models)
        win_matrix = pd.DataFrame(0, index=models, columns=models)

        for i, m_a in enumerate(models):
            for j, m_b in enumerate(models):
                if i == j:
                    continue
                for ds in pivot.index:
                    a_val = pivot.loc[ds, m_a]
                    b_val = pivot.loc[ds, m_b]
                    diff = a_val - b_val
                    if abs(diff) <= TOLERANCE:
                        pass  # tie — not counted as win for either
                    elif (higher and diff > TOLERANCE) or (not higher and diff < -TOLERANCE):
                        win_matrix.loc[m_a, m_b] += 1  # A wins

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(win_matrix, annot=True, fmt='d', cmap='Blues',
                    linewidths=0.5, ax=ax, vmin=0)
        ax.set_title(f'Win Matrix — {task_type.title()} ({metric})\n'
                     f'Cell (A, B) = # datasets where A outperforms B')
        ax.set_xlabel('Model B (loser)')
        ax.set_ylabel('Model A (winner)')
        plt.tight_layout()
        fname = f'../results/figures/win_matrix_{task_type}'
        plt.savefig(fname + '.png', dpi=SAVE_DPI, bbox_inches='tight')
        plt.savefig(fname + '.pdf', bbox_inches='tight')
        plt.show()
else:
    print("Skipping: results not available.")

## Cross-Validation Variance

Box plots per model showing variance across CV folds.

In [ ]:
if not fold_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Classification — use correct primary metrics
    clf_folds = fold_df[fold_df['task_type'].isin(['binary', 'multiclass'])].copy()
    # Prefer roc_auc for binary, log_loss for multiclass; pick whichever is available
    for metric_try in ['roc_auc', 'log_loss', 'accuracy']:
        if metric_try in clf_folds.columns and not clf_folds[metric_try].isna().all():
            sns.boxplot(data=clf_folds, x='model', y=metric_try, ax=axes[0])
            axes[0].set_title(f'Classification — {metric_try} across CV Folds')
            axes[0].tick_params(axis='x', rotation=45)
            break

    # Regression
    reg_folds = fold_df[fold_df['task_type'] == 'regression']
    if 'rmse' in reg_folds.columns:
        sns.boxplot(data=reg_folds, x='model', y='rmse', ax=axes[1])
        axes[1].set_title('Regression — RMSE across CV Folds')
        axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('../results/figures/cv_variance.png', dpi=SAVE_DPI, bbox_inches='tight')
    plt.savefig('../results/figures/cv_variance.pdf', bbox_inches='tight')
    plt.show()
else:
    print("Skipping: fold results not available.")

## Per-Dataset Results Table

All models x primary metric x mean ± std across folds.

In [ ]:
if not fold_df.empty:
    for task_type in ['binary', 'multiclass', 'regression']:
        subset = fold_df[fold_df['task_type'] == task_type]
        if subset.empty:
            continue

        metric = PRIMARY_METRICS[task_type]
        if metric not in subset.columns:
            # Try fallbacks
            for fallback in ['roc_auc', 'log_loss', 'accuracy', 'rmse']:
                if fallback in subset.columns:
                    metric = fallback
                    break
            else:
                print(f"No metric found for {task_type}, skipping.")
                continue

        summary = subset.groupby(['dataset', 'model'])[metric].agg(['mean', 'std'])
        summary['display'] = summary.apply(
            lambda r: f"{r['mean']:.4f} \u00b1 {r['std']:.4f}", axis=1)
        pivot = summary['display'].unstack('model')

        print(f"\n=== {task_type.upper()} \u2014 {metric} (mean \u00b1 std across folds) ===")
        display(pivot.style.set_caption(f'{task_type.title()} Results: {metric}'))
else:
    print("Skipping: fold results not available.")